# FT-Transformer Model for TDE Classification (v1)

**v1: Conditional Tuning Architecture - config-driven tune/fixed params**

**FT-Transformer (Feature Tokenizer + Transformer)** implementation using rtdl.

In [ ]:
# ==========================================
# CONFIGURATION & REPRODUCIBILITY
# ==========================================
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path().absolute().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config_loader import get_path, get_seed, get

RANDOM_STATE = get_seed()
MODEL_NAME = 'ft_transformer'
TUNE_MODE = get('models.ft_transformer.tune')
N_OPTUNA_TRIALS = get('models.ft_transformer.optuna_trials')

DATA_DIR = get_path('data_processed')
MODEL_DIR = get_path('models')
SUBMISSION_DIR = get_path('submissions')
TRAIN_PATH = DATA_DIR / 'further_train_processed_nn.parquet'
TEST_PATH = DATA_DIR / 'further_test_processed_nn.parquet'
FOLDS_PATH = DATA_DIR / 'train_folds.csv'

USE_SELECTED_FEATURES = True
FEATURES_JSON_PATH = get_path('features_selection') / 'selected_features.json'
FEATURES_JSON_KEY = 'final_robust_features'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {PROJECT_ROOT}')
print(f'Random State: {RANDOM_STATE}')
print(f'Tune Mode: {TUNE_MODE}')
print(f'Optuna Trials: {N_OPTUNA_TRIALS}')

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch deterministic flags set.')
print(f'Device: {DEVICE}')

In [ ]:
import re
import json
import pandas as pd
from sklearn.metrics import precision_recall_curve
from sklearn.preprocessing import StandardScaler
import optuna
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

import rtdl_revisiting_models as rtdl
print('rtdl imported successfully.')


In [ ]:
print('Loading data...')
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
folds = pd.read_csv(FOLDS_PATH)
train = train.merge(folds[['object_id', 'kfold']], on='object_id', how='left')
print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f"Class distribution: {train['target'].value_counts().to_dict()}")

In [ ]:
drop_cols = get('meta_columns')

if USE_SELECTED_FEATURES:
    print(f'Loading selected features from: {FEATURES_JSON_PATH}')
    with open(FEATURES_JSON_PATH, 'r') as f:
        features_data = json.load(f)
    selected_features = features_data[FEATURES_JSON_KEY]
    feature_cols = [c for c in selected_features if c in train.columns]
    print(f'Using {len(feature_cols)} selected features (from {len(selected_features)} in JSON)')
else:
    feature_cols = [c for c in train.columns if c not in drop_cols]
    print(f'Using all {len(feature_cols)} features')

X = train[feature_cols].values.astype(np.float32)
y = train['target'].values.astype(np.float32)
kfold = train['kfold'].values
X_test = test[feature_cols].values.astype(np.float32)
object_ids_test = test['object_id']

print(f'Feature count: {len(feature_cols)}')
print(f'X shape: {X.shape}')
print(f'X_test shape: {X_test.shape}')

In [ ]:
def train_ft_transformer(X_train, y_train, X_val, y_val, params, max_epochs=100, patience=10):
    set_seed(RANDOM_STATE)
    
    n_features = X_train.shape[1]
    common = get('models.ft_transformer.common_params')
    
    model = rtdl.FTTransformer(
        n_cont_features=n_features,
        cat_cardinalities=None,
        d_out=1,
        n_blocks=params['n_layers'],
        d_block=params['d_token'],
        attention_n_heads=common.get('n_heads', 8),
        attention_dropout=params['dropout'],
        ffn_d_hidden_multiplier=common.get('ffn_d_hidden_multiplier', 2.0),
        ffn_dropout=params['dropout'],
        residual_dropout=params['dropout'],
    ).to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=1e-5)
    criterion = torch.nn.BCEWithLogitsLoss()
    
    # Convert to Tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(DEVICE)
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1).to(DEVICE)
    
    batch_size = common.get('batch_size', 128)
    n_samples = X_train_t.shape[0]
    
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    for epoch in range(max_epochs):
        model.train()
        indices = torch.randperm(n_samples)
        
        for i in range(0, n_samples, batch_size):
            batch_idx = indices[i:i+batch_size]
            X_batch = X_train_t[batch_idx]
            y_batch = y_train_t[batch_idx]
            
            optimizer.zero_grad()
            # Pass None for categorical features
            outputs = model(X_batch, None)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_t, None)
            val_loss = criterion(val_outputs, y_val_t).item()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    
    return model

In [ ]:
def objective(trial):
    set_seed(RANDOM_STATE)
    tune_params = get('models.ft_transformer.tune_params')
    common = get('models.ft_transformer.common_params')
    
    params = {
        'n_layers': trial.suggest_int('n_layers', *tune_params['n_layers']),
        'd_token': trial.suggest_int('d_token', tune_params['d_token'][0], tune_params['d_token'][1], step=64),
        'dropout': trial.suggest_float('dropout', *tune_params['dropout']),
        'lr': trial.suggest_float('lr', tune_params['lr'][0], tune_params['lr'][1], log=True),
    }
    
    oof_preds = np.zeros(len(y))
    
    for fold in range(5):
        set_seed(RANDOM_STATE)
        
        train_idx = kfold != fold
        val_idx = kfold == fold
        
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        model = train_ft_transformer(
            X_tr, y_tr, X_val, y_val, 
            params, max_epochs=common.get('max_epochs_tuning', 50), patience=5
        )
        
        preds = predict_proba(model, X_val)
        oof_preds[val_idx] = preds
        
        del model
        torch.cuda.empty_cache()
    
    prec, rec, _ = precision_recall_curve(y, oof_preds)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-9)
    return np.max(f1)

In [ ]:
if TUNE_MODE:
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    print(f'Running Optuna with {N_OPTUNA_TRIALS} trials...')
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    best_params = study.best_params.copy()
    print(f'\nBest F1 Score: {study.best_value:.4f}')
    print(f'Best params: {best_params}')
else:
    best_params = get('models.ft_transformer.fixed_params').copy()
    print('Using fixed params from config:')
    print(best_params)

In [ ]:
print('\n=== Training Final FT-Transformer with Best Params ===')
common = get('models.ft_transformer.common_params')

oof_preds = np.zeros(len(y))
test_preds = np.zeros(len(X_test))

for fold in range(5):
    print(f'\n--- Fold {fold} ---')
    set_seed(RANDOM_STATE)
    
    train_idx = kfold != fold
    val_idx = kfold == fold
    
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    model = train_ft_transformer(
        X_tr_scaled, y_tr, X_val_scaled, y_val, 
        best_params, 
        max_epochs=common.get('max_epochs_final', 100), 
        patience=common.get('patience', 15)
    )
    
    oof_preds[val_idx] = predict_proba(model, X_val_scaled)
    test_preds += predict_proba(model, X_test_scaled) / 5
    
    print(f'Fold {fold} complete.')
    
    del model
    torch.cuda.empty_cache()

prec, rec, thresh = precision_recall_curve(y, oof_preds)
f1 = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-9)
best_thresh = thresh[np.argmax(f1)]
print(f'\nOOF F1 Score: {np.max(f1):.4f} at threshold {best_thresh:.4f}')

In [ ]:
oof_df = pd.DataFrame({'object_id': train['object_id'], 'target': y.astype(int), f'pred_{MODEL_NAME}': oof_preds})
oof_df.to_csv(MODEL_DIR / f'oof_{MODEL_NAME}.csv', index=False)
test_df = pd.DataFrame({'object_id': object_ids_test, f'pred_{MODEL_NAME}': test_preds})
test_df.to_csv(MODEL_DIR / f'preds_{MODEL_NAME}.csv', index=False)
print(f'\nSaved OOF predictions to: {MODEL_DIR}/oof_{MODEL_NAME}.csv')
print(f'Saved test predictions to: {MODEL_DIR}/preds_{MODEL_NAME}.csv')